# Experiment Run

In [3]:
#! pip install scikit-learn xgboost tqdm -q

In [4]:
import pickle

import polars as pl
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import RepeatedKFold

from scipy.sparse import csr_matrix


In [5]:

DATASET_NAME = "3_music"

In [6]:
THIS_FOLDER = Path(".")

BASE_DB_FOLDER = Path("C:\\Users\\MiguelZanchettin\\Documents\\Mestrado\\Dissertação\\Pesquisa\\research\\2_databases\\db")
TRAIN_DBS_FOLDER = BASE_DB_FOLDER / "train"
TEST_DBS_FOLDER = BASE_DB_FOLDER / "test"

DS_TRAIN_FOLDER = TRAIN_DBS_FOLDER / DATASET_NAME
DS_TEST_FOLDER = TEST_DBS_FOLDER / DATASET_NAME

DS_TRAIN_TARGET = DS_TRAIN_FOLDER / f"{DATASET_NAME}.txt"
DS_TEST_TARGET = DS_TEST_FOLDER / f"{DATASET_NAME}.txt"


## Funções auxiliares

In [7]:
def get_sparse_matrix(parquet_path: Path, embedding_strategy: str) -> csr_matrix:
    split_name = parquet_path.parent.parent.name
    handled_dfs_folder = THIS_FOLDER / "handled_dfs"
    handled_dfs_folder.mkdir(parents=True, exist_ok=True)

    handled_df_pickle_path = handled_dfs_folder / f"{DATASET_NAME}__{embedding_strategy}__{split_name}__sparse.pkl"

    if handled_df_pickle_path.exists():
        with open(handled_df_pickle_path, "rb") as f:
            return pickle.load(f)

    df = pl.read_parquet(parquet_path)

    matrix_size = df["embedding_size"].max()

    # 🔥 pega listas direto
    indices = df["embedding_indices"].to_list()
    values = df["embedding_values"].to_list()
    row_ids = df["row_index"].to_numpy()

    # 🔥 flatten manual (rápido)
    rows = np.repeat(row_ids, [len(x) for x in indices])
    cols = np.concatenate(indices)
    vals = np.concatenate(values)

    matrix = csr_matrix(
        (vals, (rows, cols)),
        shape=(df.shape[0], matrix_size),
        dtype=np.float32
    )

    with open(handled_df_pickle_path, "wb") as f:
        pickle.dump(matrix, f)

    return matrix

In [8]:
def get_dense_matrix(parquet_path: Path,  embedding_strategy: str) -> np.ndarray:
    split_name = parquet_path.parent.parent.name  # train or test
    handled_dfs_folder = THIS_FOLDER / "handled_dfs"
    handled_dfs_folder.mkdir(parents=True, exist_ok=True)

    handled_df_pickle_path = handled_dfs_folder / f"{DATASET_NAME}__{embedding_strategy}__{split_name}__dense.pkl"

    if handled_df_pickle_path.exists():
        with open(handled_df_pickle_path, "rb") as f:
            matrix = pickle.load(f)
            return matrix
        
    df = (
        pl.read_parquet(parquet_path)
        .select("row_index", "embedding")
    ) 

    vector_size = (
        df
        .with_columns(
            pl.col("embedding").list.len().alias("embedding_length")
        )
        .select("embedding_length")
        .max()
        ["embedding_length"].first()
    )
    embedding_fields = [f"emb_{i+1}" for i in range(vector_size)]

    print(f"Vector size: {vector_size}")

    matrix = (
        df
        .with_columns(
            pl.col("embedding").list.to_struct("embedding", fields=embedding_fields)
        )
        .unnest("embedding")
        .sort("row_index", descending=False)
        .drop("row_index")
        .to_numpy()
    )

    with open(handled_df_pickle_path, "wb") as f:
        pickle.dump(matrix, f)

    return matrix


In [9]:

def get_embeddings(parquet_path: Path, embedding_strategy) -> pd.DataFrame:

    if (
        "bow" in parquet_path.name 
        or "tf_idf" in parquet_path.name
    ):
        return get_sparse_matrix(
            parquet_path, 
            embedding_strategy
        )
    
    return get_dense_matrix(
        parquet_path, 
        embedding_strategy=embedding_strategy
    )

def get_target(target_path: Path) -> pd.DataFrame:
    return (
        pl.read_csv(
            target_path, 
            separator="\t",
            has_header=False,
        ).rename(
            {"column_1": "target"}
        ).with_columns(
            (pl.col("target").str.split(" ").list.get(0).cast(pl.Float64).cast(pl.Int64) - 1).alias("index"),
            pl.col("target").str.split(" ").list.get(1).cast(pl.Float64).alias("target")
        ).sort("index", descending=False)
        .drop("index")
        .to_numpy()
    )


In [10]:

def evaluate_model(dataset_name, embedding_strategy, model_name, scenario_name, model_object, X, y):
    y_pred = model_object.predict(X)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)

    return {
        "dataset_name": dataset_name,
        "embedding_strategy": embedding_strategy,
        "scenario": scenario_name,
        "model_name": model_name,
        "rmse": rmse,
        "r2": r2,
    }



## Model training functions


### 1. LinearRegression

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.feature_selection import SelectKBest, f_regression
from scipy.sparse import csr_matrix

def train_linear_regression(X_train, y_train):
    print("Training Linear Regression...")

    if isinstance(X_train, csr_matrix):
        pipeline = make_pipeline(
            Normalizer(norm='l2'),
            SelectKBest(score_func=f_regression, k=5000),
            LinearRegression()
        )
    else:
        pipeline = make_pipeline(
            StandardScaler(),
            LinearRegression()
        )

    pipeline.fit(X_train, y_train.ravel())
    return pipeline

### 2. ElasticNet (archived)

In [12]:
from sklearn.linear_model import ElasticNetCV

def train_elastic_net(X_train, y_train):
    print("Training Elastic Net...")

    elastic_net_cv = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 1.0],
        alphas=[0.1, 1.0, 10.0],
        cv=5,
        random_state=42
    )

    elastic_net_cv.fit(X_train, y_train.ravel())

    print("Best alpha:", elastic_net_cv.alpha_)
    print("Best l1_ratio:", elastic_net_cv.l1_ratio_)

    return elastic_net_cv



### 3. Ridge

In [13]:
from sklearn.linear_model import Ridge
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler, Normalizer

def train_ridge(X_train, y_train):
    print("Training Ridge...")

    if isinstance(X_train, csr_matrix):
        pipeline = make_pipeline(
            Normalizer(norm='l2'), 
            SelectKBest(score_func=f_regression, k=5000), 
            Ridge(alpha=1.0) 
        )
    else:
        pipeline = make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0)
        )

    # Sem TransformedTargetRegressor. O pipeline puro.
    pipeline.fit(X_train, y_train.ravel())

    return pipeline

### 4. Lasso

In [14]:
from sklearn.linear_model import Lasso

def train_lasso(X_train, y_train):
    print("Training Lasso...")

    if isinstance(X_train, csr_matrix):
        pipeline = make_pipeline(
            Normalizer(norm='l2'),
            SelectKBest(score_func=f_regression, k=5000),
            # Alpha reduzido para não destruir os coeficientes dos textos
            Lasso(alpha=0.001, random_state=42) 
        )
    else:
        pipeline = make_pipeline(
            StandardScaler(),
            Lasso(alpha=0.001, random_state=42)
        )

    pipeline.fit(X_train, y_train.ravel())
    return pipeline

### 5. RandomForestRegressor

In [15]:
from sklearn.ensemble import RandomForestRegressor

def train_random_forest(X_train, y_train):
    print("Training Random Forest...")

    if isinstance(X_train, csr_matrix):
        pipeline = make_pipeline(
            # Árvores não precisam de Normalizer
            SelectKBest(score_func=f_regression, k=4000), 
            RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        )
    else:
        # Sem Scaler para embeddings densos também
        pipeline = make_pipeline(
            RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        )

    pipeline.fit(X_train, y_train.ravel())
    return pipeline

### 6. XGBoost

In [16]:
from xgboost import XGBRegressor

def train_xgboost(X_train, y_train):
    print("Training XGBoost...")

    if isinstance(X_train, csr_matrix):
        pipeline = make_pipeline(
            SelectKBest(score_func=f_regression, k=4000),
            XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        )
    else:
        pipeline = make_pipeline(
            XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        )

    pipeline.fit(X_train, y_train.ravel())
    return pipeline

### 5. Deep Learning

## Experiment function

In [17]:
def sanitize_target(y_raw):
    # 1. Garante que nenhum preço seja negativo (substitui -1 por 0)
    y_clean = np.clip(y_raw, a_min=0, a_max=None)
    # 2. Converte qualquer NaN ou Inf que venha do arquivo de texto para 0
    y_clean = np.nan_to_num(y_clean, nan=0.0, posinf=0.0, neginf=0.0)
    return y_clean

In [18]:

def run_experiment(embedding_strategy, models):

    EMBEDDING_TRAIN_PATH = DS_TRAIN_FOLDER / f"{DATASET_NAME}__{embedding_strategy}.parquet"
    EMBEDDING_TEST_PATH = DS_TEST_FOLDER / f"{DATASET_NAME}__{embedding_strategy}.parquet"

    objects_folder = THIS_FOLDER / "objects"
    objects_folder.mkdir(parents=True, exist_ok=True)

    # Carrega dados de treino
    X_train = get_embeddings(EMBEDDING_TRAIN_PATH, embedding_strategy)
    y_train_raw = get_target(DS_TRAIN_TARGET)
    
    if DATASET_NAME in ("1_walmes", "3_music"):
        y_train_raw = sanitize_target(y_train_raw)
        y_train = np.log1p(y_train_raw)
        del y_train_raw

    else:
        y_train = y_train_raw
        del y_train_raw

    print("Train shapes:", f"X={X_train.shape}, y={y_train.shape}")

    # --- CV robusta: RepeatedKFold 5x3 ---
    cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
    cv_splits = list(cv.split(X_train))

    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    
    print(f"Running CV ({len(cv_splits)} folds)...")
    cv_raw_results = []


    for fold_idx, (train_idx, val_idx) in enumerate(tqdm(cv_splits, desc="CV")):
        repeat = fold_idx // 5
        fold = fold_idx % 5

        if isinstance(X_train, csr_matrix):
            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
        else:
            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]

        y_fold_train = y_train[train_idx]
        y_fold_val = y_train[val_idx]

        for model in models:
            model_instance = model["train_function"](X_fold_train, y_fold_train)
            y_pred = model_instance.predict(X_fold_val)
            rmse = np.sqrt(mean_squared_error(y_fold_val, y_pred))
            r2 = r2_score(y_fold_val, y_pred)

            cv_raw_results.append({
                "dataset_name": DATASET_NAME,
                "embedding_strategy": embedding_strategy,
                "model_name": model["model_name"],
                "repeat": repeat,
                "fold": fold,
                "rmse": rmse,
                "r2": r2,
                "scenario": "cv",
            })

    cv_raw_df = pd.DataFrame(cv_raw_results)
    with open(objects_folder / f"{DATASET_NAME}__{embedding_strategy}__cv_raw.pkl", "wb") as f:
        pickle.dump(cv_raw_df, f)

    # Resumo da CV
    cv_summary_df = (
        cv_raw_df
        .groupby(["dataset_name", "embedding_strategy", "model_name"])
        .agg(
            rmse_mean=("rmse", "mean"),
            rmse_std=("rmse", "std"),
            r2_mean=("r2", "mean"),
            r2_std=("r2", "std"),
        )
        .reset_index()
    )
    with open(objects_folder / f"{DATASET_NAME}__{embedding_strategy}__cv_summary.pkl", "wb") as f:
        pickle.dump(cv_summary_df, f)

    print("CV summary:")
    print(cv_summary_df.to_string(index=False))

    del X_fold_train, X_fold_val, y_fold_train, y_fold_val, cv_raw_results

    # --- Treino final no conjunto completo + avaliação no teste holdhout ---
    print("\nFitting final models on full training set...")
    X_test = get_embeddings(EMBEDDING_TEST_PATH, embedding_strategy)
    y_test_raw = get_target(DS_TEST_TARGET)
    
    if DATASET_NAME in ("1_walmes", "3_music"):
        y_test_raw = sanitize_target(y_test_raw)
        y_test = np.log1p(y_test_raw)
        del y_test_raw
    else:
        y_test = y_test_raw
        del y_test_raw

    print("Test shapes:", f"X={X_test.shape}, y={y_test.shape}")

    test_results = []
    for model in tqdm(models, desc="Final fit + test eval"):
        model_instance = model["train_function"](X_train, y_train)
        models[models.index(model)]["model_instance"] = model_instance
        result = evaluate_model(
            DATASET_NAME,
            embedding_strategy,
            model["model_name"],
            "test",
            model_instance,
            X_test,
            y_test,
        )
        test_results.append(result)

    test_results_df = pd.DataFrame(test_results)
    with open(objects_folder / f"{DATASET_NAME}__{embedding_strategy}__test_results.pkl", "wb") as f:
        pickle.dump(test_results_df, f)

    del X_train, y_train, X_test, y_test, test_results, test_results_df

    print("Done!")


## Run experiment

In [19]:

models = [
    {"model_name": "Linear Regression", "train_function": train_linear_regression},
    {"model_name": "Ridge",             "train_function": train_ridge},
    {"model_name": "Lasso",             "train_function": train_lasso},
    {"model_name": "Random Forest",     "train_function": train_random_forest},
    {"model_name": "XGBoost",           "train_function": train_xgboost},
]

embedding_strategies = [
    "bow",
    "glove",
    "n_gram_bow",
    "roberta",
    "tf_idf",
    "word2vec",
    # "openai" # O tamanho dos embeddings da OpenAI não permite o tamanho dos textos que necessito
]

for embedding_strategy in embedding_strategies:
    print(f"\n{'='*60}")
    print(f"Embedding strategy: {embedding_strategy}")
    print(f"{'='*60}")
    run_experiment(embedding_strategy, models)



Embedding strategy: bow
Train shapes: X=(44463, 57395), y=(44463, 1)
X_train: (44463, 57395)
y_train: (44463, 1)
Running CV (15 folds)...


CV:   0%|          | 0/15 [00:00<?, ?it/s]

Training Linear Regression...
Training Ridge...
Training Lasso...
Training Random Forest...
Training XGBoost...


CV:   7%|▋         | 1/15 [01:49<25:30, 109.35s/it]

Training Linear Regression...
Training Ridge...
Training Lasso...
Training Random Forest...
Training XGBoost...


...

Training Random Forest...


Final fit + test eval:  80%|████████  | 4/5 [06:12<02:09, 129.61s/it]

Training XGBoost...


Final fit + test eval: 100%|██████████| 5/5 [06:21<00:00, 76.36s/it] 

Done!


In [20]:

objects_folder = THIS_FOLDER / "objects"
object_files = sorted(objects_folder.glob(f"{DATASET_NAME}*test_results.pkl"))

dfs = []
for obj_file in object_files:
    print(f"Found: {obj_file.name}")
    dfs.append(pd.read_pickle(obj_file))

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
display(df)


Found: 3_music__bow__test_results.pkl
Found: 3_music__glove__test_results.pkl
Found: 3_music__n_gram_bow__test_results.pkl
Found: 3_music__roberta__test_results.pkl
Found: 3_music__tf_idf__test_results.pkl
Found: 3_music__word2vec__test_results.pkl


,dataset_name,embedding_strategy,scenario,model_name,rmse,r2
0,3_music,bow,test,Linear Regression,2.304471,0.135322
1,3_music,bow,test,Ridge,2.226933,0.192530
2,3_music,bow,test,Lasso,2.412808,0.052111
3,3_music,bow,test,Random Forest,1.948751,0.381664
4,3_music,bow,test,XGBoost,2.303461,0.136080
5,3_music,glove,test,Linear Regression,2.426026,0.041697
6,3_music,glove,test,Ridge,2.426025,0.041698
7,3_music,glove,test,Lasso,2.425703,0.041952
8,3_music,glove,test,Random Forest,1.972218,0.366682
9,3_music,glove,test,XGBoost,2.190915,0.218439
